# Catalyst distance analysis

How far apart are five phosphine catalysts **from each other**, and how far is each from the
**Kraken population** of 1,223 monodentate organophosphorus ligands, measured separately in
each of the project's feature sets.

Two different questions, two different tools:

| Question | Measure | Package |
|---|---|---|
| How similar are these five to each other? | pairwise euclidean / cosine, hierarchical clustering | `sklearn.metrics.pairwise_distances`, `scipy.cluster.hierarchy` |
| Is a catalyst typical of Kraken, or an outlier? | leverage, Mahalanobis, k-NN distance | `numpy`, `sklearn.covariance`, `sklearn.neighbors` |

The second group is the standard **applicability domain (AD)** toolkit from the QSAR
literature -- the accepted way to ask whether a query compound is close enough to a reference
set for a model trained on that set to be trusted:

* **Leverage** `h = x'(X'X)^-1 x`, the classic Williams-plot measure. It is proportional to
  the Mahalanobis distance from the training centroid. The usual warning threshold is
  `h* = 3(M+1)/N` for M descriptors and N reference compounds.
* **Mahalanobis distance** to the Kraken centroid, which accounts for descriptor covariance
  rather than treating every direction as equally important. Reported both from the ordinary
  covariance and from `MinCovDet` (robust, so a handful of extreme ligands cannot inflate it).
* **k-NN distance**, the mean distance to the k nearest Kraken ligands -- a local measure that
  catches a compound sitting in a sparse region even if it is near the global centroid.
  Reported as a percentile of the Kraken population so the number is interpretable.

Everything is computed on **standardised** descriptors (`StandardScaler` fitted on all 1,223
Kraken ligands), because the raw descriptors are in different units -- percent, Debye, eV --
and euclidean distance on raw values would just measure whichever column has the largest range.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# The Kraken table and the fitted reference PCA live in the hazel project; every
# project shares them unrefitted, so the feature sets mean the same thing here.
PROJECT = Path.cwd() / "gp_collab_hazel"
if not PROJECT.exists():
    PROJECT = Path.cwd().parent / "gp_collab_hazel"
sys.path.insert(0, str(PROJECT))

from gpc.config import TWO, FIVE          # noqa: E402
from gpc.data import load_bundle          # noqa: E402

reactions, ligands, reference, prepared = load_bundle(PROJECT / "inputs")
print(f"Kraken reference: {len(ligands)} ligands x {len(reference.columns)} descriptors")

## The catalysts and the feature sets

Kraken ids verified against `kraken_identifiers.csv` by name and canonical SMILES.
`ligand_ohe` is deliberately absent: it has no descriptors, so distance is undefined -- every
ligand is equidistant from every other by construction.

In [ ]:
CATALYSTS = {
    "(tBu)3P":   8,     # PtBu3      -- trialkyl
    "XPhos":     1,     # Xphos      -- dialkylbiaryl
    "SPhos":     3,     # SPhos      -- dialkylbiaryl
    "RuPhos":    4,     # RuPhos     -- dialkylbiaryl
    "BrettPhos": 102,   # BrettPhos  -- dialkylbiaryl
}

FEATURE_SETS = {
    "selected_2": list(TWO),
    "selected_5": list(FIVE),
    "pc_top":     list(reference.top_features),
    "pc_scores":  ["PC1", "PC2", "PC3", "PC4"],
}

missing = [n for n, i in CATALYSTS.items() if i not in set(ligands.kraken_id)]
assert not missing, f"not in the Kraken table: {missing}"

for name, cols in FEATURE_SETS.items():
    shown = cols if len(cols) <= 5 else cols[:4] + ["..."]
    print(f"  {name:11s} {len(cols):2d} descriptors: {shown}")

In [ ]:
def standardised(cols):
    """Kraken descriptors -> mean-imputed, z-scored on the full population.

    The scaler is fitted on all 1,223 ligands, not on the five queries, so a
    distance means "how unusual is this within Kraken".
    """
    X = ligands[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    X = pd.DataFrame(SimpleImputer(strategy="mean").fit_transform(X),
                     columns=cols, index=ligands.kraken_id)
    Z = pd.DataFrame(StandardScaler().fit_transform(X.values), columns=cols, index=X.index)
    query = Z.loc[list(CATALYSTS.values())].copy()
    query.index = list(CATALYSTS)
    return Z, query


def population_distance(Z, query, k=5):
    """Applicability-domain measures for each query against the Kraken population."""
    M, N = Z.shape[1], len(Z)

    # Leverage, the Williams-plot measure. pinv keeps it stable if the descriptor
    # block is rank-deficient (pc_top holds near-duplicate size measures).
    gram = np.linalg.pinv(Z.values.T @ Z.values)
    leverage = np.einsum("ij,jk,ik->i", query.values, gram, query.values)
    h_star = 3 * (M + 1) / N

    maha_emp = np.sqrt(EmpiricalCovariance().fit(Z.values).mahalanobis(query.values))
    # random_state fixed so the robust fit is reproducible run to run
    maha_rob = np.sqrt(MinCovDet(random_state=0).fit(Z.values).mahalanobis(query.values))

    nn = NearestNeighbors(n_neighbors=k + 1).fit(Z.values)
    d_query, _ = nn.kneighbors(query.values)
    d_ref, _ = nn.kneighbors(Z.values)
    knn_query = d_query[:, 1:].mean(axis=1)      # column 0 is the point itself
    knn_ref = d_ref[:, 1:].mean(axis=1)
    percentile = [100.0 * (knn_ref < v).mean() for v in knn_query]

    return pd.DataFrame({
        "leverage": leverage,
        "outside_AD": leverage > h_star,
        "mahalanobis": maha_emp,
        "mahalanobis_robust": maha_rob,
        f"knn{k}_distance": knn_query,
        "knn_percentile": percentile,
    }, index=query.index), h_star

## 1. How far apart are the five from each other?

Euclidean distance in standardised descriptor space. Cosine is shown alongside because it
compares *direction* rather than magnitude -- two ligands can be similar in character while
differing in overall size, and cosine will say so where euclidean will not.

In [ ]:
plt.close("all")
results = {}

fig, axes = plt.subplots(2, len(FEATURE_SETS), figsize=(4.2 * len(FEATURE_SETS), 7.6),
                         constrained_layout=True)
for j, (name, cols) in enumerate(FEATURE_SETS.items()):
    Z, query = standardised(cols)
    results[name] = {"Z": Z, "query": query}
    for i, metric in enumerate(("euclidean", "cosine")):
        D = pd.DataFrame(pairwise_distances(query.values, metric=metric),
                         index=query.index, columns=query.index)
        results[name][metric] = D
        ax = axes[i, j]
        ax.imshow(D.values, cmap="viridis")
        ax.set_xticks(range(len(D)), D.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(D)), D.index, fontsize=7)
        hi = D.values.max() * 0.6
        for a in range(len(D)):
            for b in range(len(D)):
                ax.text(b, a, f"{D.values[a, b]:.2f}", ha="center", va="center",
                        fontsize=6, color="w" if D.values[a, b] < hi else "k")
        ax.set_title(f"{name}\n{metric}", fontsize=9)
fig_pairwise = fig
display(fig_pairwise)
plt.close(fig_pairwise)

In [ ]:
for name in FEATURE_SETS:
    D = results[name]["euclidean"]
    print(f"=== {name} -- euclidean, standardised ===")
    print(D.round(2).to_string())
    iu = np.triu_indices_from(D.values, 1)
    pairs = sorted(zip(D.values[iu], [(D.index[a], D.columns[b]) for a, b in zip(*iu)]))
    print(f"  closest : {pairs[0][1][0]} / {pairs[0][1][1]}  ({pairs[0][0]:.2f})")
    print(f"  farthest: {pairs[-1][1][0]} / {pairs[-1][1][1]}  ({pairs[-1][0]:.2f})\n")

### Hierarchical clustering

Average-linkage dendrogram on the euclidean distances. This is the compact way to see which
catalysts group together, and whether that grouping survives changing the feature set.

In [ ]:
plt.close("all")
fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(4.0 * len(FEATURE_SETS), 3.4),
                         constrained_layout=True)
axl = np.atleast_1d(axes)
for ax, name in zip(axl, FEATURE_SETS):
    D = results[name]["euclidean"]
    Zl = linkage(squareform(D.values, checks=False), method="average")
    dendrogram(Zl, labels=list(D.index), ax=ax, leaf_font_size=8)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("distance" if ax is axl[0] else "")
    ax.tick_params(axis="x", rotation=45)
fig_dendro = fig
display(fig_dendro)
plt.close(fig_dendro)

## 2. How far is each from the Kraken population?

`leverage` above `h*` is the conventional flag for "outside the applicability domain".
`knn_percentile` is the most directly readable number: 90 means this catalyst sits in a
sparser neighbourhood than 90% of Kraken ligands.

In [ ]:
ad = {}
for name, cols in FEATURE_SETS.items():
    table, h_star = population_distance(results[name]["Z"], results[name]["query"])
    ad[name] = table
    print(f"=== {name}   (M={len(cols)}, N={len(results[name]['Z'])}, h* = {h_star:.4f}) ===")
    print(table.round(3).to_string(), "\n")

In [ ]:
summary = pd.DataFrame({name: t["knn_percentile"] for name, t in ad.items()})
print("k-NN distance percentile within Kraken (higher = more unusual)")
print(summary.round(1).to_string())

plt.close("all")
ax = summary.plot.bar(figsize=(8.5, 4), width=0.8, edgecolor="white", linewidth=0.6)
ax.set(ylabel="k-NN distance percentile", xlabel="")
ax.axhline(90, color="0.4", linestyle="--", linewidth=1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="feature set", fontsize=8, title_fontsize=8, frameon=False,
          loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.spines[["top", "right"]].set_visible(False)
fig_ad = ax.figure
fig_ad.tight_layout()
display(fig_ad)
plt.close(fig_ad)

## 3. Where they sit inside Kraken

A 2-component PCA of each feature set, with the whole Kraken population in grey and the five
catalysts marked. This is for orientation only -- two components cannot show the full geometry
of a 12-dimensional set, and the percentages in the axis labels say how much is being lost.

In [ ]:
plt.close("all")
fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(4.0 * len(FEATURE_SETS), 3.8),
                         constrained_layout=True)
for ax, name in zip(np.atleast_1d(axes), FEATURE_SETS):
    Z, query = results[name]["Z"], results[name]["query"]
    pca = PCA(n_components=2).fit(Z.values)
    P, Q = pca.transform(Z.values), pca.transform(query.values)
    ax.scatter(P[:, 0], P[:, 1], s=6, c="0.82", edgecolors="none")
    ax.scatter(Q[:, 0], Q[:, 1], s=55, c="#c0392b", edgecolors="k", linewidths=0.5, zorder=3)
    for (x, y), lab in zip(Q, query.index):
        ax.annotate(lab, (x, y), fontsize=7, xytext=(4, 4), textcoords="offset points")
    v = pca.explained_variance_ratio_
    ax.set(title=name, xlabel=f"PC1 ({v[0]:.0%})", ylabel=f"PC2 ({v[1]:.0%})")
    ax.spines[["top", "right"]].set_visible(False)
fig_pca = fig
display(fig_pca)
plt.close(fig_pca)

## 4. Does the answer depend on the feature set?

If the five catalysts rank the same way in every feature set, the conclusion is robust. Where
they disagree, the feature set is doing the talking -- worth knowing before quoting any single
number.

In [ ]:
rank = pd.DataFrame({name: t["knn_percentile"].rank(ascending=False).astype(int)
                     for name, t in ad.items()})
print("rank by k-NN percentile (1 = most unusual within Kraken)")
print(rank.to_string())
print("\nspread of ranks per catalyst (0 = every feature set agrees):")
print((rank.max(axis=1) - rank.min(axis=1)).to_string())

print("\ncorrelation of the pairwise-distance matrices between feature sets")
iu = np.triu_indices(len(CATALYSTS), 1)
flat = pd.DataFrame({n: results[n]["euclidean"].values[iu] for n in FEATURE_SETS})
print(flat.corr(method="spearman").round(3).to_string())

## 5. The Kraken PC1 vs PC2 map

The PCA above was refitted *per feature set*, so its axes differ from panel to panel. This
section instead uses **Kraken's own principal components** -- the 4-component PCA fitted on
all 190 standardised descriptors across all 1,223 ligands, stored in `pca_reference` and
shipped precomputed as the `PC1..PC4` columns. These are the same axes every project uses for
the `pc_scores` feature set, and the same map the Kraken authors plot.

So this is one fixed chemical space, not a per-feature-set view: a ligand's position here does
not change with which descriptors a model happens to use.

In [ ]:
# PC1..PC4 are precomputed in ligand_features.csv and are exactly
# (standardise -> centre -> project onto reference.components), verified elsewhere.
PCS = ligands.set_index("kraken_id")[["PC1", "PC2", "PC3", "PC4"]]
var = np.asarray(reference.variance_ratio)
print("Kraken reference PCA, explained variance:",
      "  ".join(f"PC{i+1} {v:.1%}" for i, v in enumerate(var)),
      f"| first two = {var[:2].sum():.1%}")

plt.close("all")
fig, ax = plt.subplots(figsize=(7.2, 6.0), constrained_layout=True)
ax.scatter(PCS.PC1, PCS.PC2, s=7, c="0.84", edgecolors="none", label=f"Kraken ({len(PCS)})")

q = PCS.loc[list(CATALYSTS.values())]
ax.scatter(q.PC1, q.PC2, s=95, c="#c0392b", edgecolors="k", linewidths=0.7, zorder=3,
           label="catalysts of interest")
for (kid, row), lab in zip(q.iterrows(), CATALYSTS):
    ax.annotate(lab, (row.PC1, row.PC2), fontsize=9, fontweight="bold",
                xytext=(7, 5), textcoords="offset points")

ax.axhline(0, color="0.6", lw=0.6, zorder=0)
ax.axvline(0, color="0.6", lw=0.6, zorder=0)
ax.set(xlabel=f"PC1 ({var[0]:.1%} of descriptor variance)",
       ylabel=f"PC2 ({var[1]:.1%})",
       title="Kraken chemical space -- reference PCA over 190 descriptors")
ax.legend(frameon=False, fontsize=9, loc="best")
ax.spines[["top", "right"]].set_visible(False)
fig_kraken_pca = fig
display(fig_kraken_pca)
plt.close(fig_kraken_pca)

print()
print("coordinates on the Kraken axes:")
coords = q.copy(); coords.index = list(CATALYSTS)
print(coords.round(3).to_string())

### What the axes mean

From the loadings, the four reference components separate into interpretable blocks -- bulk
size, local geometry at phosphorus, conformational flexibility, and electronics at phosphorus.
The three highest-|loading| descriptors on PC1 and PC2 are printed below so the map can be
read chemically rather than just geometrically.

In [ ]:
C = np.asarray(reference.components)
cols = list(reference.columns)
for k in (0, 1):
    order = np.argsort(-np.abs(C[k]))[:6]
    print(f"PC{k+1} ({var[k]:.1%}) top loadings:")
    for j in order:
        print(f"    {C[k, j]:+.3f}  {cols[j]}")
    print()

## 6. Distance heat maps for the catalysts in each dataset

The five above are a hand-picked set. This section instead takes the catalysts **actually
screened in each benchmark** and shows how far apart they are, in every feature set.

| dataset | group column | catalysts |
|---|---|---|
| Perera 2018 (Suzuki) | `ligand` | 8 |
| Ahneman/Doyle 2018 (Buchwald-Hartwig) | `ligand` | 4 |
| Cernak Suzuki informer | `catalyst` | 5 |

Reading these: the numbers are euclidean distances in standardised Kraken units, so a value of
1.0 means the two ligands differ by about one Kraken standard deviation, averaged in
quadrature over that feature set's descriptors. Dark blocks are chemically similar ligands;
a dataset whose heat map is mostly dark is asking its model to extrapolate over a narrow
range, and one with bright off-diagonal blocks spans more of the space.

In [ ]:
DATASETS = {
    "Perera (Suzuki, 8)": {
        "AmPhos": 216, "CataCXium A": 10, "P(Cy)3": 11, "P(Ph)3": 17,
        "P(o-Tol)3": 9, "P(tBu)3": 8, "SPhos": 3, "XPhos": 1,
    },
    "Doyle BH (4)": {
        "AdBrettPhos": 347, "XPhos": 1, "t-BuBrettPhos": 89, "t-BuXPhos": 90,
    },
    "Cernak (5)": {
        "Aphos G3": 216, "RuPhos G3": 4, "Xphos G3": 1,
        "tBu3P G2": 8, "tBuXphos G3": 90,
    },
}

for label, mapping in DATASETS.items():
    absent = [n for n, i in mapping.items() if i not in set(ligands.kraken_id)]
    assert not absent, f"{label}: not in Kraken -> {absent}"
print("every dataset catalyst resolves in the Kraken table")


def dataset_distances(mapping, cols, metric="euclidean"):
    """Pairwise distances for one dataset's catalysts, standardised on all of Kraken."""
    Z, _ = standardised(cols)
    sub = Z.loc[list(mapping.values())].copy()
    sub.index = list(mapping)
    return pd.DataFrame(pairwise_distances(sub.values, metric=metric),
                        index=sub.index, columns=sub.index)

In [ ]:
plt.close("all")
nrow, ncol = len(DATASETS), len(FEATURE_SETS)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.5 * nrow),
                         constrained_layout=True)
dataset_D = {}

for i, (label, mapping) in enumerate(DATASETS.items()):
    dataset_D[label] = {}
    for j, (fname, cols) in enumerate(FEATURE_SETS.items()):
        D = dataset_distances(mapping, cols)
        dataset_D[label][fname] = D
        ax = axes[i, j]
        im = ax.imshow(D.values, cmap="magma")
        ax.set_xticks(range(len(D)), D.columns, rotation=90, fontsize=6)
        ax.set_yticks(range(len(D)), D.index, fontsize=6)
        if len(D) <= 6:
            hi = D.values.max() * 0.55
            for a in range(len(D)):
                for b in range(len(D)):
                    ax.text(b, a, f"{D.values[a, b]:.1f}", ha="center", va="center",
                            fontsize=5.5,
                            color="w" if D.values[a, b] < hi else "k")
        ax.set_title(f"{label}\n{fname}", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=6)

fig_dataset_heat = fig
display(fig_dataset_heat)
plt.close(fig_dataset_heat)

In [ ]:
# How much of the space does each screen actually span?
rows = []
for label, per_set in dataset_D.items():
    for fname, D in per_set.items():
        iu = np.triu_indices_from(D.values, 1)
        d = D.values[iu]
        rows.append({"dataset": label, "feature_set": fname, "n_catalysts": len(D),
                     "mean_pairwise": d.mean(), "min_pairwise": d.min(),
                     "max_pairwise": d.max()})
spread = pd.DataFrame(rows)
print("pairwise distance spread, standardised Kraken units")
print(spread.pivot(index="dataset", columns="feature_set", values="mean_pairwise").round(2).to_string())
print()
for label, per_set in dataset_D.items():
    D = per_set["selected_5"]
    iu = np.triu_indices_from(D.values, 1)
    pairs = sorted(zip(D.values[iu], [(D.index[a], D.columns[b]) for a, b in zip(*iu)]))
    print(f"{label} (selected_5): closest {pairs[0][1][0]}/{pairs[0][1][1]} = {pairs[0][0]:.2f}"
          f"   farthest {pairs[-1][1][0]}/{pairs[-1][1][1]} = {pairs[-1][0]:.2f}")

In [ ]:
# Every dataset catalyst on the Kraken map, coloured by which screen used it
plt.close("all")
fig, ax = plt.subplots(figsize=(7.6, 6.2), constrained_layout=True)
ax.scatter(PCS.PC1, PCS.PC2, s=7, c="0.88", edgecolors="none", label=f"Kraken ({len(PCS)})")
colors = {"Perera (Suzuki, 8)": "#c0392b", "Doyle BH (4)": "#1f77b4", "Cernak (5)": "#2ca02c"}
markers = {"Perera (Suzuki, 8)": "o", "Doyle BH (4)": "s", "Cernak (5)": "^"}
for label, mapping in DATASETS.items():
    q = PCS.loc[list(mapping.values())]
    ax.scatter(q.PC1, q.PC2, s=80, c=colors[label], marker=markers[label],
               edgecolors="k", linewidths=0.6, zorder=3, label=label, alpha=0.85)
seen = {}
for label, mapping in DATASETS.items():
    for name, kid in mapping.items():
        if kid in seen:
            continue
        seen[kid] = name
        row = PCS.loc[kid]
        ax.annotate(name, (row.PC1, row.PC2), fontsize=6.5,
                    xytext=(5, 4), textcoords="offset points")
ax.axhline(0, color="0.6", lw=0.6, zorder=0); ax.axvline(0, color="0.6", lw=0.6, zorder=0)
ax.set(xlabel=f"PC1 ({var[0]:.1%})", ylabel=f"PC2 ({var[1]:.1%})",
       title="Catalysts screened in each benchmark, on the Kraken axes")
ax.legend(frameon=False, fontsize=8, loc="best")
ax.spines[["top", "right"]].set_visible(False)
fig_dataset_pca = fig
display(fig_dataset_pca)
plt.close(fig_dataset_pca)

overlap = {k: v for k, v in seen.items()
           if sum(k in m.values() for m in DATASETS.values()) > 1}
print("catalysts shared by more than one dataset:",
      {seen[k]: k for k in overlap} or "none")

## Save

Nothing is written until `SAVE = True`.

In [ ]:
SAVE = False
OUT = Path.cwd() / "exports/catalyst_distance"

if SAVE:
    OUT.mkdir(parents=True, exist_ok=True)
    for name, t in ad.items():
        t.to_csv(OUT / f"applicability_{name}.csv")
        results[name]["euclidean"].to_csv(OUT / f"pairwise_euclidean_{name}.csv")
    summary.to_csv(OUT / "knn_percentile_summary.csv")
    for label, per_set in dataset_D.items():
        for fs, D in per_set.items():
            tag = label.split()[0].lower()
            D.to_csv(OUT / f"dataset_{tag}_{fs}.csv")
    spread.to_csv(OUT / "dataset_pairwise_spread.csv", index=False)
    for fname, fig in (("pairwise", fig_pairwise), ("dendrogram", fig_dendro),
                       ("applicability", fig_ad), ("pca", fig_pca),
                       ("kraken_pc1_pc2", fig_kraken_pca),
                       ("dataset_heatmaps", fig_dataset_heat),
                       ("dataset_pca", fig_dataset_pca)):
        for ext in ("png", "pdf"):
            fig.savefig(OUT / f"{fname}.{ext}", dpi=300, bbox_inches="tight")
    print("Saved:", OUT)
else:
    print("Preview only. Set SAVE=True when ready.")